# Soninké TTS Libre — génération en lot et sauvegarde persistante

Ce notebook reprend une liste CSV de mots/phrases et génère les audios l'un après l'autre. Les fichiers et le journal sont conservés dans Google Drive pour survivre à une réinitialisation de Colab. Il reprend au prochain élément après une interruption.

**Important :** le contrôle automatique vérifie les fichiers et quelques propriétés techniques, pas l'exactitude linguistique. Chaque audio reste « à valider » jusqu'à l'écoute du référent soninké. Ne pas ajouter d'audios privés au dépôt public. Les poids OmniVoice sont soumis à des restrictions non commerciales : vérifier les licences avant toute diffusion ou usage commercial.

In [ ]:
%pip -q install omnivoice soundfile pandas

## 1. Connecter Google Drive

Drive est le dossier de sauvegarde durable. Il ne faut pas enregistrer les résultats uniquement dans `/content`, qui est temporaire.

In [ ]:
from google.colab import drive, files
from pathlib import Path
import os, json, time, csv, zipfile, datetime
import numpy as np
import pandas as pd
import soundfile as sf
from IPython.display import Audio, display

drive.mount('/content/drive')
PROJECT_DIR = Path('/content/drive/MyDrive/SONINKE_TTS_LIBRE')
AUDIO_DIR = PROJECT_DIR / 'audio'
AUDIO_DIR.mkdir(parents=True, exist_ok=True)
print('Dossier durable :', PROJECT_DIR)
print('Audios :', AUDIO_DIR)


## 2. Préparer la voix de référence

Le meilleur résultat de nos essais précédents utilisait les enregistrements d'origine du locuteur (`Daga goli.wav` et `Li taxu.wav`) combinés en `reference_soninke.wav`. Si la référence n'est pas dans Drive, téléverse les deux enregistrements originaux depuis ton iPhone dans la cellule suivante. Ne remplace pas les originaux par des audios générés si tu peux l'éviter.

In [ ]:
ref_path = PROJECT_DIR / 'reference_soninke.wav'
ref_text = 'Daga goli. Li taxu.'

if not ref_path.exists():
    print('Téléverse les enregistrements originaux Daga goli.wav et Li taxu.wav, ou un reference_soninke.wav déjà créé.')
    uploaded = files.upload()
    for name, data in uploaded.items():
        (PROJECT_DIR / name).write_bytes(data)
    direct_ref = PROJECT_DIR / 'reference_soninke.wav'
    if direct_ref.exists():
        ref_path = direct_ref
    else:
        a_path = PROJECT_DIR / 'Daga goli.wav'
        b_path = PROJECT_DIR / 'Li taxu.wav'
        if a_path.exists() and b_path.exists():
            a, sr_a = sf.read(a_path)
            b, sr_b = sf.read(b_path)
            if sr_a != sr_b:
                raise ValueError(f'Les fréquences audio diffèrent : {sr_a} et {sr_b} Hz. Convertis-les à la même fréquence avant de continuer.')
            if a.ndim > 1: a = a.mean(axis=1)
            if b.ndim > 1: b = b.mean(axis=1)
            silence = np.zeros(int(0.4 * sr_a), dtype=np.float32)
            combined = np.concatenate([a.astype(np.float32), silence, b.astype(np.float32)])
            sf.write(ref_path, combined, sr_a)
        else:
            raise FileNotFoundError('Référence manquante. Téléverse les deux fichiers originaux nommés exactement Daga goli.wav et Li taxu.wav, ou reference_soninke.wav.')

print('Référence :', ref_path, '| transcription :', ref_text)
display(Audio(str(ref_path)))


## 3. Constituer automatiquement une liste lexicale sourcée

Le notebook récupère les données linguistiques `snk_engine`, dont `LICENSE-DATA` autorise la réutilisation sous CC BY 4.0 avec attribution. Il extrait les formes de verbes, pronoms, objets, compléments et mots uniques rencontrés dans les corpus CC BY 4.0. **Ce n'est pas un dictionnaire exhaustif de tout le soninké** : c'est une base lexicale sourcée à élargir et à faire vérifier par le référent. La source décrit sa variété de référence et ne représente pas nécessairement toutes les variétés.

Les six phrases testées par le référent sont ajoutées comme tests privés distincts. Le fichier généré reste dans Google Drive ; aucun audio privé n'est envoyé à GitHub.

In [ ]:
%pip -q install pyyaml
import urllib.request, re, yaml, json, csv, hashlib

WORDLIST = PROJECT_DIR / 'wordlist.csv'
SOURCE_LICENSE = 'CC BY 4.0'
SOURCE_ATTRIBUTION = 'noah-medra (ORCID 0009-0003-6018-9802), snk_engine (2026)'
BASE = 'https://raw.githubusercontent.com/oscarmedra/snk_engine/49f50a939ded51c5ffe7db404c45f40fc75a3c88/'
SOURCES = {
    'grammar': BASE + 'data/grammaire/soninke.yaml',
    'objects': BASE + 'data/grammaire/objets.yaml',
    'license': BASE + 'LICENSE-DATA',
}
CORPUS_FILES = [
    'corpus/valide/serie_02.jsonl', 'corpus/valide/serie_03.jsonl',
    'corpus/valide/texte_journee.jsonl', 'corpus/valide/texte_marche.jsonl',
    'corpus/valide/texte_pluies.jsonl', 'corpus/valide/texte_repas.jsonl',
    'corpus/valide/texte_travail.jsonl', 'corpus/valide/texte_village.jsonl',
    'corpus/valide/validation_01.jsonl', 'corpus/valide/validation_02.jsonl',
    'corpus/valide/validation_03.jsonl'
]

def get_text(url):
    with urllib.request.urlopen(url, timeout=30) as response:
        return response.read().decode('utf-8')

license_text = get_text(SOURCES['license'])
if 'Creative Commons Attribution 4.0 International (CC BY 4.0)' not in license_text or 'data/' not in license_text or 'corpus/' not in license_text:
    raise RuntimeError('La licence des données source n’a pas été confirmée comme attendu. Arrêt sans importer les données.')
grammar = yaml.safe_load(get_text(SOURCES['grammar']))
objects = yaml.safe_load(get_text(SOURCES['objects']))
entries = {}

def add_entry(text, source, license_name, note=''):
    text = str(text).strip()
    if not text:
        return
    key = text.casefold()
    if key not in entries:
        entries[key] = {'text': text, 'source': source, 'license': license_name, 'note': note}

# Lexique morphologique déclaré dans la grammaire : pronoms et formes verbales attestées.
for item in grammar.get('pronoms', []) or []:
    add_entry(item.get('forme'), SOURCES['grammar'], SOURCE_LICENSE, 'forme sourcée; orthographe à vérifier par le référent')
for verb_id, item in (grammar.get('verbes', {}) or {}).items():
    for field in ('forme', 'nom_action', 'gerondif'):
        if item.get(field):
            add_entry(item[field], SOURCES['grammar'], SOURCE_LICENSE, f'forme verbale {field}; orthographe à vérifier par le référent')
for section in ('objets', 'complements'):
    for item in objects.get(section, []) or []:
        add_entry(item.get('snk'), SOURCES['objects'], SOURCE_LICENSE, f'entrée {section}; orthographe à vérifier par le référent')

# Corpus parallèle : inclure chaque token distinct ainsi que les phrases source entières.
token_pattern = re.compile(r"[\w]+(?:['’][\w]+)*", re.UNICODE)
for relative_path in CORPUS_FILES:
    url = BASE + relative_path
    try:
        for line in get_text(url).splitlines():
            if not line.strip():
                continue
            record = json.loads(line)
            sentence = str(record.get('soninke', '')).strip()
            if not sentence:
                continue
            for token in token_pattern.findall(sentence):
                add_entry(token, url, SOURCE_LICENSE, 'mot extrait d’une phrase source; contexte perdu; validation requise')
    except Exception as exc:
        print(f'Attention : corpus non importé {relative_path}: {exc}')

# Les tests privés validés précédemment restent inclus, mais ne sont pas redistribués comme données ouvertes.
private_phrases = ['Daga goli', 'Li taxu', 'Daga iige', 'Daga sohu', 'Mpai reni', 'Mpai telle']
for phrase in private_phrases:
    add_entry(phrase, 'phrase de test fournie et validée par le référent', 'usage privé du projet uniquement; redistribution non autorisée', 'test TTS déjà écouté par le référent')

rows = []
for i, (key, item) in enumerate(sorted(entries.items()), start=1):
    rows.append({'id': f'snk-{i:05d}', 'text': item['text'], 'source': item['source'], 'license': item['license'], 'attribution': SOURCE_ATTRIBUTION if item['license'] == SOURCE_LICENSE else 'Référent Soninké; usage privé uniquement', 'note': item['note']})
generated_wordlist = PROJECT_DIR / 'wordlist_from_open_source.csv'
pd.DataFrame(rows).to_csv(generated_wordlist, index=False, encoding='utf-8-sig')

# Ne jamais écraser une liste personnalisée. Première exécution : utiliser la liste source générée.
if not WORDLIST.exists():
    pd.DataFrame(rows).to_csv(WORDLIST, index=False, encoding='utf-8-sig')
    print('Liste de travail initiale créée depuis la source CC BY 4.0.')
else:
    print('Liste wordlist.csv existante conservée, sans écrasement.')

df = pd.read_csv(WORDLIST, dtype=str).fillna('')
required = {'id', 'text', 'source', 'license'}
missing = required - set(df.columns)
if missing:
    raise ValueError(f'Colonnes manquantes dans wordlist.csv : {sorted(missing)}')
df = df[df['text'].str.strip() != ''].copy()
if df['id'].duplicated().any():
    raise ValueError('Les identifiants de la colonne id doivent être uniques.')
print(f'{len(df)} entrées chargées depuis {WORDLIST}')
print(f'{len(rows)} entrées lexicales/phrases sourcées préparées dans {generated_wordlist}')
display(df[['id','text','source','license']].head(20))
print('La source CC BY 4.0 est attribuée par entrée. Les mots extraits de phrases ont perdu leur contexte et nécessitent une validation humaine.')


## 4. Charger OmniVoice sur GPU

Si Colab ne propose pas de GPU gratuit, arrête sans souscrire à un forfait.

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Aucun GPU disponible. Arrête sans payer et réessaie lorsque le GPU gratuit est disponible.')
from omnivoice import OmniVoice
model = OmniVoice.from_pretrained('k2-fsa/OmniVoice', device_map='cuda:0', dtype=torch.float16)
if not any(str(x).lower() == 'soninke' for x in model.supported_language_names()):
    raise RuntimeError('La langue soninke ne figure plus dans les langues prises en charge par cette version du modèle.')
SAMPLE_RATE = 24000  # fréquence utilisée lors des générations de référence validées
print('Modèle prêt. GPU :', torch.cuda.get_device_name(0), '| fréquence audio :', SAMPLE_RATE)


## 5. Générer en lot, reprendre après interruption

Chaque fichier est écrit directement dans Google Drive. Si le notebook s'interrompt, relance cette cellule : les WAV déjà présents ne seront pas régénérés. Les statuts de validation humaine restent « en attente ».

In [ ]:
MANIFEST = PROJECT_DIR / 'manifest.csv'
records = []
if MANIFEST.exists():
    old = pd.read_csv(MANIFEST, dtype=str).fillna('')
    old_by_id = {row['id']: row for _, row in old.iterrows()}
else:
    old_by_id = {}

for _, row in df.iterrows():
    item_id, phrase = row['id'].strip(), row['text'].strip()
    safe_id = ''.join(c if c.isalnum() or c in '-_' else '_' for c in item_id)
    wav_path = AUDIO_DIR / f'{safe_id}.wav'
    previous = old_by_id.get(item_id, {})
    if wav_path.exists() and wav_path.stat().st_size > 1000:
        status = 'audio_present_reused'
        err = ''
    else:
        status, err = 'generated', ''
        try:
            print(f'Génération {item_id}: {phrase}')
            result = model.generate(text=phrase, language='soninke', ref_audio=str(ref_path), ref_text=ref_text, num_step=32)
            if result is None or len(result) == 0 or len(result[0]) == 0:
                raise RuntimeError('Le modèle n’a retourné aucun échantillon audio.')
            sf.write(str(wav_path), result[0], SAMPLE_RATE)
            print('Enregistré dans Drive :', wav_path.name)
        except Exception as exc:
            status, err = 'generation_error', f'{type(exc).__name__}: {exc}'
            print('ERREUR — génération interrompue pour cette entrée :', err)
    duration = ''
    peak = ''
    if wav_path.exists():
        try:
            info = sf.info(str(wav_path))
            duration = round(info.duration, 3)
            audio_data, _ = sf.read(str(wav_path), always_2d=True)
            peak = round(float(np.max(np.abs(audio_data))) if audio_data.size else 0.0, 5)
        except Exception as exc:
            status, err = 'technical_check_error', str(exc)
    records.append({
        'id': item_id, 'text': phrase, 'source': row['source'], 'license': row['license'], 'attribution': row.get('attribution', ''),
        'audio_file': str(wav_path) if wav_path.exists() else '', 'technical_status': status,
        'duration_seconds': duration, 'peak_amplitude': peak, 'generation_error': err,
        'human_pronunciation_validation': previous.get('human_pronunciation_validation', 'pending'),
        'human_notes': previous.get('human_notes', '')
    })
    pd.DataFrame(records).to_csv(MANIFEST, index=False, encoding='utf-8-sig')

print('Traitement terminé ou interrompu. Manifeste sauvegardé dans Drive :', MANIFEST)
display(pd.DataFrame(records)[['id','text','technical_status','duration_seconds','human_pronunciation_validation']])


## 6. Contrôle technique et écoute groupée

Le rapport détecte les audios manquants et les erreurs de génération. Il ne peut pas certifier automatiquement qu'un mot est bien écrit ou bien prononcé en soninké. Pour réduire la fatigue, cette cellule crée aussi un seul long fichier audio de révision avec de courtes pauses entre les entrées. Écoute-le en suivant le même ordre dans le manifeste CSV ; note ensuite les corrections dans `human_pronunciation_validation` et `human_notes`.

In [ ]:
manifest_df = pd.read_csv(MANIFEST, dtype=str).fillna('')
playlist = PROJECT_DIR / 'playlist.html'
parts = ['<!doctype html><html lang="fr"><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>Validation audio Soninké</title><body><h1>Validation audio Soninké</h1><p>Écouter chaque fichier et noter la validation dans manifest.csv. La génération n’équivaut pas à une validation linguistique.</p>']
audio_segments = []
review_rows = []
silence = np.zeros(int(0.8 * SAMPLE_RATE), dtype=np.float32)
for _, r in manifest_df.iterrows():
    if r.get('audio_file') and Path(r['audio_file']).exists():
        audio_path = Path(r['audio_file'])
        rel = audio_path.name
        parts.append(f'<section><h2>{r["text"]}</h2><p>ID : {r["id"]} — Statut : {r["human_pronunciation_validation"]}</p><audio controls preload="none" src="audio/{rel}"></audio></section><hr>')
        try:
            clip, sr = sf.read(str(audio_path), dtype='float32', always_2d=False)
            if sr != SAMPLE_RATE:
                print(f'Attention : {rel} est à {sr} Hz ; il est exclu du fichier groupé qui attend {SAMPLE_RATE} Hz.')
            else:
                if getattr(clip, 'ndim', 1) > 1:
                    clip = clip.mean(axis=1)
                if len(clip):
                    audio_segments.append(clip.astype(np.float32))
                    audio_segments.append(silence)
                    review_rows.append({'id': r['id'], 'text': r['text'], 'audio_file': rel})
        except Exception as exc:
            print(f'Attention : impossible de lire {rel}: {exc}')
parts.append('</body></html>')
playlist.write_text('\n'.join(parts), encoding='utf-8')
review_audio = PROJECT_DIR / 'review_playlist.wav'
if audio_segments:
    combined = np.concatenate(audio_segments)
    sf.write(str(review_audio), combined, SAMPLE_RATE)
    print(f'Audio groupé créé : {review_audio} — {len(review_rows)} entrées, {len(combined)/SAMPLE_RATE/60:.1f} minute(s)')
    display(Audio(str(review_audio)))
else:
    print('Aucun WAV disponible pour la playlist audio groupée.')
print('Manifeste CSV à garder :', MANIFEST)
print('Liste audio HTML :', playlist)
print('Nombre d’audios présents :', sum(bool(x) and Path(x).exists() for x in manifest_df['audio_file']))
print('Erreurs de génération :', int((manifest_df['technical_status'] == 'generation_error').sum()))
display(manifest_df[['id','text','technical_status','duration_seconds','human_pronunciation_validation','human_notes']])


## 7. Télécharger une archive de sauvegarde

L'archive est créée à partir du dossier durable Google Drive. Le téléchargement est une copie supplémentaire, pas le seul emplacement de sauvegarde.

In [ ]:
zip_path = PROJECT_DIR / 'soninke_tts_batch_backup.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for p in PROJECT_DIR.rglob('*'):
        if p.is_file() and p != zip_path:
            zf.write(p, arcname=str(p.relative_to(PROJECT_DIR)))
print('Archive sauvegardée dans Drive :', zip_path)
files.download(str(zip_path))
